In [1]:
import requests
import json
import anthropic
import pandas as pd
from pprint import pprint
from sklearn.metrics import precision_recall_fscore_support
from sklearn.preprocessing import MultiLabelBinarizer
from pathlib import Path
from collections import Counter

In [2]:
#LABEL_STUDIO_URL = "http://localhost:8080"
##API_TOKEN = ""
#PROJECT_ID = 3

In [3]:
#headers = {
#    "Authorization": f"Token {API_TOKEN}",
#    "Content-Type": "application/json"
#}

In [4]:
 #predictions ophalen
#  "http://localhost:8080/api/predictions?project=3&limit=10000",
#    headers=headers
#).json()
#
#predictions = response if isinstance(response, list) else response.get('results', [])
#
#print(f"Aantal predictions: {len(predictions)}")
#print(f"Predictions met niet-leeg result: {sum(1 for p in predictions if p.get('result'))}")
#
#with open('../../data/H8/llm/results_llm.json', 'w') as f:
#    json.dump(predictions, f)

In [5]:
with open('../../data/H8/llm/geluid_annotaties_predictions.json', 'r') as f:
    results_llm = json.load(f)

In [6]:
LABELS = {
    "doorbell ringing": "deurbellen",
    "singing": "zingen",
    "making noise": "lawaai maken",
    "disrupting nightrest": "nachtrust verstoren",
    "playing inn music": "herbergmuziek",
    "scolding": "schelden",
    "yelling": "roepen",
    "arguing": "ruzie maken",
    "playing street music": "straatmuziek",
    "traffic noise": "verkeersgeluid",
    "other sound": "ander geluid",
}
EVAL = [l for l in LABELS if l != "other sound"]

In [7]:
def spans(blocks, zonder=()):
    return [res for b in blocks[:1] for res in b["result"] if set(res["value"]["labels"]) - set(zonder)]

for naam, zonder in [("alle labels", ()), ("zonder ander geluid", ("other sound",))]:
    print(naam)
    for key, tekst in [("predictions", "voorspeld"), ("annotations", "gecorrigeerd")]:
        n = [len(spans(r[key], zonder)) for r in results_llm]
        print(f"  {tekst:<13} {sum(n)} vermeldingen in {sum(x > 0 for x in n)} gebeurtenissen")

alle labels
  voorspeld     3730 vermeldingen in 2804 gebeurtenissen
  gecorrigeerd  2756 vermeldingen in 1712 gebeurtenissen
zonder ander geluid
  voorspeld     2859 vermeldingen in 2146 gebeurtenissen
  gecorrigeerd  2515 vermeldingen in 1570 gebeurtenissen


In [8]:
nagekeken = [r for r in results_llm if r["annotations"]]
niet_nagekeken = sum(bool(spans(r["predictions"])) and not r["annotations"] for r in results_llm)
print("gebeurtenissen met voorspelling maar zonder correctie:", niet_nagekeken)

gebeurtenissen met voorspelling maar zonder correctie: 0


In [9]:
def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0
    r = tp / (tp + fn) if tp + fn else 0
    return p, r, 2 * p * r / (p + r) if p + r else 0

def present(blocks):
    return {l for res in spans(blocks) for l in res["value"]["labels"]}

pairs = [(present(r["annotations"]), present(r["predictions"])) for r in nagekeken]

rows, tot = [], Counter()
for lab in EVAL:
    tp = sum(lab in g and lab in p for g, p in pairs)
    fp = sum(lab in p and lab not in g for g, p in pairs)
    fn = sum(lab in g and lab not in p for g, p in pairs)
    tot.update(tp=tp, fp=fp, fn=fn)
    rows.append([LABELS[lab], tp + fp, tp + fn, *prf(tp, fp, fn)])

tabel = pd.DataFrame(rows, columns=["label", "voorspellingen", "correcties", "precisie", "recall", "F1"])
tabel = tabel.sort_values("F1", ascending=False).reset_index(drop=True)
tabel.loc[len(tabel)] = ["macrogemiddelde", tabel["voorspellingen"].sum(), tabel["correcties"].sum(),
                         *tabel[["precisie", "recall", "F1"]].mean()]
tabel.loc[len(tabel)] = ["microgemiddelde", tot["tp"] + tot["fp"], tot["tp"] + tot["fn"],
                         *prf(tot["tp"], tot["fp"], tot["fn"])]

In [10]:
tabel[["voorspellingen", "correcties"]] = tabel[["voorspellingen", "correcties"]].astype(int)
tabel.round(3).to_csv("../../tab/H8/8.3_evaluatie.csv", index=False, sep=";", decimal=",")
tabel.style.format(precision=3, decimal=",", thousands=".").hide(axis="index")

label,voorspellingen,correcties,precisie,recall,F1
deurbellen,213,180,"0,803","0,950","0,870"
zingen,152,205,"0,961","0,712","0,818"
lawaai maken,530,422,"0,711","0,893","0,792"
nachtrust verstoren,459,447,"0,765","0,785","0,775"
herbergmuziek,66,79,"0,833","0,696","0,759"
schelden,108,189,"0,926","0,529","0,673"
roepen,921,514,"0,496","0,889","0,637"
ruzie maken,111,226,"0,901","0,442","0,593"
straatmuziek,27,21,"0,519","0,667","0,583"
verkeersgeluid,29,34,"0,414","0,353","0,381"
